In [4]:
import kagglehub

kagglehub.login()

In [5]:
path = kagglehub.competition_download('deep-learning-for-computer-vision-and-nlp-2026-10-1')
print('Шлях до файлів:', path)

100%|██████████| 997M/997M [00:13<00:00, 80.3MB/s] 

Extracting files...


Шлях до файлів: /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-10-1


In [10]:
!pip -q install kagglehub lightgbm scikit-learn pandas numpy scipy pillow tqdm

import os
import glob
import gc
import random
import warnings
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from PIL import Image
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import kagglehub

DATA_DIR = '/root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-10-1'

if not os.path.exists(os.path.join(DATA_DIR, 'train.csv')):
    DATA_DIR = kagglehub.competition_download('deep-learning-for-computer-vision-and-nlp-2026-10-1')

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

IMAGE_ROOT = os.path.join(DATA_DIR, 'images', 'images')
TRAIN_IMAGE_DIR = os.path.join(IMAGE_ROOT, 'train')
TEST_IMAGE_DIR = os.path.join(IMAGE_ROOT, 'test')

print('Розмір train:', train.shape)
print('Розмір test:', test.shape)
print('Папка з даними:', DATA_DIR)
print('Колонки train:', train.columns.tolist())

assert 'PetID' in train.columns and 'Description' in train.columns and 'AdoptionSpeed' in train.columns
assert 'PetID' in test.columns and 'Description' in test.columns

Розмір train: (6431, 3)
Розмір test: (1891, 2)
Папка з даними: /root/.cache/kagglehub/competitions/deep-learning-for-computer-vision-and-nlp-2026-10-1
Колонки train: ['PetID', 'Description', 'AdoptionSpeed']


In [ ]:
#готуємо текст
train['Description'] = train['Description'].fillna('').astype(str)
test['Description'] = test['Description'].fillna('').astype(str)

all_text = pd.concat([train['Description'], test['Description']], ignore_index=True)

word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=40000,
    min_df=2,
    sublinear_tf=True,
    strip_accents='unicode',
    dtype=np.float32)

char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(3, 5),
    max_features=20000,
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32)

X_word = word_vectorizer.fit_transform(all_text)
X_char = char_vectorizer.fit_transform(all_text)
X_text = hstack([X_word, X_char], format='csr', dtype=np.float32)

print('Текстові ознаки готові:', X_text.shape)

#рахуємо прості характеристики опису
text_stats = np.zeros((len(all_text), 8), dtype=np.float32)

for i, description in enumerate(all_text):
    words = description.split()
    text_stats[i] = [
        len(description),
        len(words),
        len(set(words)),
        description.count(' '),
        description.count('!'),
        description.count('?'),
        description.count('.'),
        sum(char.isdigit() for char in description)]

#знаходимо фотографії кожної тварини
def collect_images(folder):
    result = {}
    if not os.path.isdir(folder):
        return result

    for path in glob.glob(os.path.join(folder, '*')):
        if not os.path.isfile(path):
            continue
        filename = os.path.basename(path)
        pet_id = filename.split('-')[0]
        result.setdefault(pet_id, []).append(path)

    for pet_id in result:
        result[pet_id] = sorted(result[pet_id])[:3]

    return result

train_images = collect_images(TRAIN_IMAGE_DIR)
test_images = collect_images(TEST_IMAGE_DIR)
all_images = [train_images, test_images]

print('Тварин з фотографіями у train:', len(train_images))
print('Тварин з фотографіями у test:', len(test_images))

#завантажуємо модель для отримання ознак з фото
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input

image_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    pooling='avg',
    input_shape=(160, 160, 3))

image_features = np.zeros((len(all_text), 1280), dtype=np.float32)

#обробляємо фотографії пакетами
image_records = []

for split_index, (frame, image_dict) in enumerate(zip([train, test], all_images)):
    offset = 0 if split_index == 0 else len(train)

    for row_index, pet_id in enumerate(frame['PetID'].astype(str)):
        for path in image_dict.get(pet_id, []):
            image_records.append((offset + row_index, path))

batch_size = 32

for start in tqdm(range(0, len(image_records), batch_size), desc='Обробка фотографій'):
    batch_records = image_records[start:start + batch_size]
    batch_images = []
    batch_rows = []

    for row_index, path in batch_records:
        try:
            with Image.open(path) as img:
                img = img.convert('RGB').resize((160, 160))
                batch_images.append(np.asarray(img, dtype=np.float32))
                batch_rows.append(row_index)
        except Exception:
            continue

    if batch_images:
        batch_array = preprocess_input(np.stack(batch_images))
        features = image_model.predict(batch_array, verbose=0)

        for row_index, feature in zip(batch_rows, features):
            image_features[row_index] += feature

#усереднюємо ознаки за кількістю фото
image_counts = np.zeros(len(all_text), dtype=np.float32)

for row_index, _ in image_records:
    image_counts[row_index] += 1

image_counts = np.maximum(image_counts, 1)

image_features /= image_counts[:, None]

print('Ознаки фотографій готові:', image_features.shape)

del image_model
gc.collect()

Текстові ознаки готові: (8322, 60000)
Тварин з фотографіями у train: 6431
Тварин з фотографіями у test: 1899
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Обробка фотографій:   0%|          | 0/657 [00:00<?, ?it/s]

Ознаки фотографій готові: (8322, 1280)


35980

In [ ]:
from scipy.optimize import differential_evolution

y = train['AdoptionSpeed'].astype(int).to_numpy()
n_train = len(train)

train_idx, val_idx = train_test_split(
    np.arange(n_train),
    test_size=0.2,
    random_state=SEED,
    stratify=y)

#стискаємо текстові ознаки
n_components = min(180, X_text.shape[1] - 1, len(train_idx) - 1)

text_svd = TruncatedSVD(n_components=n_components, n_iter=7, random_state=SEED)
text_svd.fit(X_text[train_idx])

text_train_svd = text_svd.transform(X_text[train_idx]).astype(np.float32)
text_val_svd = text_svd.transform(X_text[val_idx]).astype(np.float32)

#об'єднуємо текст, фото та характеристики опису
X_model_train = np.hstack([
    text_train_svd,
    image_features[train_idx],
    text_stats[train_idx]]).astype(np.float32)

X_model_val = np.hstack([
    text_val_svd,
    image_features[val_idx],
    text_stats[val_idx]]).astype(np.float32)

model = LGBMRegressor(
    objective='regression',
    n_estimators=1200,
    learning_rate=0.025,
    num_leaves=20,
    max_depth=-1,
    min_child_samples=25,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_lambda=3.0,
    verbosity=-1,
    random_state=SEED,
    n_jobs=-1)

model.fit(X_model_train, y[train_idx])

val_pred = model.predict(X_model_val)

#підбираємо пороги для п'яти класів
def score_thresholds(thresholds):
    thresholds = np.sort(thresholds)
    pred_classes = np.digitize(val_pred, thresholds)
    return -cohen_kappa_score(
        y[val_idx],
        pred_classes,
        labels=[0, 1, 2, 3, 4],
        weights='quadratic')

result = differential_evolution(
    score_thresholds,
    bounds=[(0.0, 4.0)] * 4,
    seed=SEED,
    maxiter=40,
    popsize=8,
    polish=True)

best_thresholds = np.sort(result.x)
val_classes = np.digitize(val_pred, best_thresholds)

qwk = cohen_kappa_score(
    y[val_idx],
    val_classes,
    labels=[0, 1, 2, 3, 4],
    weights='quadratic')

print('QWK на валідації:', round(qwk, 4))
print('Знайдені пороги:', np.round(best_thresholds, 4))
print('Розподіл прогнозів:', np.bincount(val_classes, minlength=5))

QWK на валідації: 0.4657
Знайдені пороги: [1.4627 2.1971 2.6561 3.05  ]
Розподіл прогнозів: [ 20 211 328 386 342]


In [ ]:
#навчаємо текстове стиснення на всьому train
n_components = min(180, X_text.shape[1] - 1, n_train - 1)

final_svd = TruncatedSVD(n_components=n_components, n_iter=7, random_state=SEED)
final_svd.fit(X_text[:n_train])

text_train_final = final_svd.transform(X_text[:n_train]).astype(np.float32)
text_test_final = final_svd.transform(X_text[n_train:]).astype(np.float32)

#об'єднуємо всі ознаки
X_final_train = np.hstack([
    text_train_final,
    image_features[:n_train],
    text_stats[:n_train]]).astype(np.float32)

X_final_test = np.hstack([
    text_test_final,
    image_features[n_train:],
    text_stats[n_train:]]).astype(np.float32)

#навчаємо фінальну модель
final_model = LGBMRegressor(
    objective='regression',
    n_estimators=1200,
    learning_rate=0.025,
    num_leaves=20,
    max_depth=-1,
    min_child_samples=25,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_lambda=3.0,
    verbosity=-1,
    random_state=SEED,
    n_jobs=-1)

final_model.fit(X_final_train, y)

test_pred = final_model.predict(X_final_test)
test_classes = np.digitize(test_pred, best_thresholds).astype(int)

submission = pd.DataFrame({
    'PetID': test['PetID'],
    'AdoptionSpeed': test_classes})

submission.to_csv('submission.csv', index=False)

print('Файл submission.csv створено')
print('Розмір файлу:', submission.shape)
print('Розподіл прогнозів:', submission['AdoptionSpeed'].value_counts().sort_index().to_dict())
print(submission.head())

from google.colab import files
files.download('submission.csv')

Файл submission.csv створено
Розмір файлу: (1891, 2)
Розподіл прогнозів: {0: 31, 1: 345, 2: 445, 3: 464, 4: 606}
       PetID  AdoptionSpeed
0  6697a7f62              3
1  23b64fe21              3
2  41e824cbe              4
3  6c3d7237b              4
4  97b0b5d92              3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ВИСНОВОК

Під час цієї роботи я навчилася використовувати машинне навчання для прогнозування швидкості адопції тварин я спробувала поєднати текстові описи та фотографії, щоб модель могла враховувати більше інформації про кожну тварину

Також я декілька разів намагалася покращити результат і підвищити QWK на валідації. Для цього я змінювала параметри моделі, пробувала різні підходи до обробки тексту та фотографій, поєднувала різні ознаки й підбирала пороги для прогнозування класів. Деякі варіанти давали кращий результат, але значного покращення досягти не вдалося

У результаті найкращий показник QWK на валідації, якого мені вдалося досягти, становив приблизно 0,49. Я зрозуміла, що якість моделі залежить не тільки від кількості ознак, а й від того, наскільки добре вони підходять для конкретного завдання. Попри те, що результат не такий високий, як хотілося б, я отримала практичний досвід підготовки даних, навчання моделі, оцінювання її якості та пошуку способів покращення прогнозів